In [2]:
import pandas as pd
import os

# Define the path to your quarter folder
data_path = "2025_yearly/2026q1/"
files = ['sub.txt', 'num.txt', 'pre.txt', 'tag.txt']

print(f"--- PREVIEWING FIRST 10 COLUMNS FOR {data_path} ---")

for file_name in files:
    file_full_path = os.path.join(data_path, file_name)
    
    # Check if file exists to avoid errors
    if os.path.exists(file_full_path):
        # Load only 5 rows and as many columns as exist (up to 10)
        df_preview = pd.read_csv(file_full_path, sep='\t', nrows=5, low_memory=False)
        
        print(f"\n>>> FILE: {file_name}")
        # iloc[:, :10] gets all rows and columns from index 0 to 9
        display(df_preview.iloc[:, :10])
    else:
        print(f"\n>>> FILE: {file_name} NOT FOUND.")

--- PREVIEWING FIRST 10 COLUMNS FOR 2025_yearly/2026q1/ ---

>>> FILE: sub.txt


,adsh,cik,name,sic,countryba,stprba,cityba,zipba,bas1,bas2
0,0000006955-26-000025,6955,ENERPAC TOOL GROUP CORP,3590,US,WI,MILWAUKEE,53203-2917,"648 N PLANKINTON AVE, 4TH FLOOR",NaN
1,0000014693-26-000010,14693,BROWN FORMAN CORP,2080,US,KY,LOUISVILLE,40210,850 DIXIE HWY,NaN
2,0000014846-26-000007,14846,BRT APARTMENTS CORP.,6798,US,NY,GREAT NECK,11021-3190,60 CUTTER MILL RD,SUITE 303
3,0000015847-26-000003,15847,BUTLER NATIONAL CORP,7990,US,KS,NEW CENTURY,66031,ONE AERO PLAZA,NaN
4,0000016732-26-000006,16732,CAMPBELL'S CO,2000,US,NJ,CAMDEN,08103,CAMPBELL PL,NaN



>>> FILE: num.txt


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote
0,0000002488-26-000018,AccumulatedOtherComprehensiveIncomeLossNetOfTax,us-gaap/2025,20241231,0,USD,NaN,NaN,-6.900000e+07,NaN
1,0000002488-26-000018,AdjustmentForAmortization,us-gaap/2025,20251231,4,USD,NaN,NaN,2.254000e+09,NaN
2,0000002488-26-000018,AdjustmentsToAdditionalPaidInCapitalWarrantIssued,us-gaap/2025,20241231,4,USD,EquityComponents=AdditionalPaidInCapital;,NaN,0.000000e+00,NaN
3,0000002488-26-000018,CashCashEquivalentsRestrictedCashAndRestricted...,us-gaap/2025,20221231,0,USD,NaN,NaN,4.835000e+09,NaN
4,0000002488-26-000018,CashCashEquivalentsRestrictedCashAndRestricted...,us-gaap/2025,20231231,0,USD,NaN,NaN,3.933000e+09,NaN



>>> FILE: pre.txt


,adsh,report,line,stmt,inpth,rfile,tag,version,plabel,negating
0,0000002488-26-000018,3,1,IS,0,H,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2025,Net revenue,0
1,0000002488-26-000018,3,2,IS,0,H,CostOfGoodsAndServiceExcludingDepreciationDepl...,us-gaap/2025,Cost of sales,0
2,0000002488-26-000018,3,3,IS,0,H,AmortizationOfAcquisitionRelatedIntangiblesCOGS,0000002488-26-000018,Amortization of acquisition-related intangible...,0
3,0000002488-26-000018,3,4,IS,0,H,CostOfGoodsAndServicesSold,us-gaap/2025,Total cost of sales,0
4,0000002488-26-000018,3,5,IS,0,H,GrossProfit,us-gaap/2025,Gross profit,0



>>> FILE: tag.txt


,tag,version,custom,abstract,datatype,iord,crdr,tlabel,doc
0,AccountsAndNotesReceivableNet,us-gaap/2024,0,0,monetary,I,D,"Accounts and Financing Receivable, after Allow...","Amount, after allowance for credit loss, of ac..."
1,AccountsAndOtherReceivablesNetCurrent,us-gaap/2024,0,0,monetary,I,D,"Accounts and Other Receivables, Net, Current","Amount, after allowance, receivable from custo..."
2,AccountsNotesAndLoansReceivableNetCurrent,us-gaap/2024,0,0,monetary,I,D,"Accounts and Financing Receivable, after Allow...","Amount, after allowance for credit loss, of ac..."
3,AccountsPayableAndAccruedLiabilitiesCurrent,us-gaap/2024,0,0,monetary,I,C,"Accounts Payable and Accrued Liabilities, Current",Sum of the carrying values as of the balance s...
4,AccountsPayableAndAccruedLiabilitiesCurrentAnd...,us-gaap/2024,0,0,monetary,I,C,Accounts Payable and Accrued Liabilities,Sum of the carrying values as of the balance s...


In [ ]:
import pandas as pd
import os

# 1. Define Paths
base_path = "2025_yearly/2026q1/"

# 2. Load all files with all columns (no 'usecols' to ensure no compromise)
print("Loading all files...")
sub = pd.read_csv(os.path.join(base_path, "sub.txt"), sep='\t', low_memory=False)
num = pd.read_csv(os.path.join(base_path, "num.txt"), sep='\t', low_memory=False)
pre = pd.read_csv(os.path.join(base_path, "pre.txt"), sep='\t', low_memory=False)
tag = pd.read_csv(os.path.join(base_path, "tag.txt"), sep='\t', low_memory=False)

# 3. Step-by-Step Merging
print("Merging data into Master File...")

# A. Link Numbers to their Definitions (NUM + TAG)
# Joined on 'tag' and 'version'
master = pd.merge(num, tag, on=['tag', 'version'], how='outer', suffixes=('', '_tag_file'))

# B. Link to Presentation Map (Master + PRE)
# Joined on 'adsh', 'tag', and 'version'
master = pd.merge(master, pre, on=['adsh', 'tag', 'version'], how='outer', suffixes=('', '_pre_file'))

# C. Link to Company Identity (Master + SUB)
# Joined on 'adsh'
master = pd.merge(master, sub, on='adsh', how='outer', suffixes=('', '_sub_file'))

# 4. Filter for Northern Trust (CIK: 73124)
# This keeps the file manageable while showing you the result
ntrs_master = master[master['cik'] == 73124]

# 5. Save to a single Master CSV
ntrs_master.to_csv("NTRS_2026_Master_Data.csv", index=False)

print(f"Success! Master file created with {ntrs_master.shape[1]} columns and {len(ntrs_master)} rows.")
display(ntrs_master.head())

Loading all files...


In [ ]:
import pandas as pd

# 1. Load your existing master CSV
df = pd.read_csv('NTRS_2026_Master_Data.csv')

# 2. Export to Excel
# Note: You may need to install openpyxl first: pip install openpyxl
df.to_excel('NTRS_2026_Master_Data.xlsx', index=False, sheet_name='NTRS_Big_Data')

print("Conversion complete! Your Excel file is ready.")

Conversion complete! Your Excel file is ready.


In [ ]:
df = pd.read_csv("NTRS_2026_Master_Data.csv")

In [ ]:
df.head()

,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,...,period,fy,fp,filed,accepted,prevrpt,detail,instance,nciks,aciks
0,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20241231,4,USD,NaN,NaN,35100000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
1,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20231231,4,USD,NaN,NaN,-3000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
2,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20251231,4,USD,NaN,NaN,67800000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
3,0000073124-26-000016,AccumulatedOtherComprehensiveIncomeLossNetOfTax,us-gaap/2025,20241231,0,USD,NaN,NaN,-814000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
4,0000073124-26-000016,AccumulatedOtherComprehensiveIncomeLossNetOfTax,us-gaap/2025,20241231,0,USD,ConsolidatedEntities=ParentCompany;,NaN,-814000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN


In [ ]:
df.describe()

,ddate,qtrs,coreg,value,custom,abstract,report,line,inpth,negating,...,changed,wksi,fye,period,fy,filed,prevrpt,detail,nciks,aciks
count,1.268000e+03,1268.000000,0.0,1.268000e+03,1268.000000,1268.0,1268.000000,1268.000000,1268.000000,1268.000000,...,1268.0,1268.0,1268.0,1268.0,1268.0,1268.0,1268.0,1268.0,1268.0,0.0
mean,2.024255e+07,1.958991,NaN,4.843198e+09,0.064669,0.0,5.209779,20.408517,0.056782,0.083596,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN
std,8.262684e+03,2.000368,NaN,1.810094e+10,0.246038,0.0,2.208221,12.702049,0.231518,0.276891,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
min,2.022123e+07,0.000000,NaN,-2.016990e+10,0.000000,0.0,3.000000,1.000000,0.000000,0.000000,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN
25%,2.024123e+07,0.000000,NaN,9.850000e+06,0.000000,0.0,3.000000,12.000000,0.000000,0.000000,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN
50%,2.024123e+07,0.000000,NaN,4.086000e+08,0.000000,0.0,5.000000,18.000000,0.000000,0.000000,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN
75%,2.025123e+07,4.000000,NaN,2.031100e+09,0.000000,0.0,7.000000,25.000000,0.000000,0.000000,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN
max,2.025123e+07,4.000000,NaN,1.771327e+11,1.000000,0.0,9.000000,52.000000,1.000000,1.000000,...,19780525.0,1.0,1231.0,20251231.0,2025.0,20260224.0,0.0,1.0,1.0,NaN


In [ ]:
df.isnull().sum()

adsh             0
tag              0
version          0
ddate            0
qtrs             0
uom              0
segments       368
coreg         1268
value            0
footnote      1266
custom           0
abstract         0
datatype         0
iord             0
crdr            37
tlabel           0
doc              0
report           0
line             0
stmt             0
inpth            0
rfile            0
plabel           0
negating         0
cik              0
name             0
sic              0
countryba        0
stprba           0
cityba           0
zipba            0
bas1             0
bas2          1268
baph             0
countryma        0
stprma           0
cityma           0
zipma            0
mas1             0
mas2          1268
countryinc       0
stprinc          0
ein              0
former           0
changed          0
afs              0
wksi             0
fye              0
form             0
period           0
fy               0
fp               0
filed       

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1268 entries, 0 to 1267
Data columns (total 59 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   adsh        1268 non-null   str    
 1   tag         1268 non-null   str    
 2   version     1268 non-null   str    
 3   ddate       1268 non-null   int64  
 4   qtrs        1268 non-null   int64  
 5   uom         1268 non-null   str    
 6   segments    900 non-null    str    
 7   coreg       0 non-null      float64
 8   value       1268 non-null   float64
 9   footnote    2 non-null      str    
 10  custom      1268 non-null   int64  
 11  abstract    1268 non-null   int64  
 12  datatype    1268 non-null   str    
 13  iord        1268 non-null   str    
 14  crdr        1231 non-null   str    
 15  tlabel      1268 non-null   str    
 16  doc         1268 non-null   str    
 17  report      1268 non-null   float64
 18  line        1268 non-null   float64
 19  stmt        1268 non-null   str    
 2

In [ ]:
df[df.isnull().any(axis=1)]

,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,...,period,fy,fp,filed,accepted,prevrpt,detail,instance,nciks,aciks
0,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20241231,4,USD,NaN,NaN,35100000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
1,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20231231,4,USD,NaN,NaN,-3000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
2,0000073124-26-000016,AccretionAmortizationOfDiscountsAndPremiumsInv...,us-gaap/2025,20251231,4,USD,NaN,NaN,67800000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
3,0000073124-26-000016,AccumulatedOtherComprehensiveIncomeLossNetOfTax,us-gaap/2025,20241231,0,USD,NaN,NaN,-814000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
4,0000073124-26-000016,AccumulatedOtherComprehensiveIncomeLossNetOfTax,us-gaap/2025,20241231,0,USD,ConsolidatedEntities=ParentCompany;,NaN,-814000000.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,0000073124-26-000016,WeightedAverageNumberOfDilutedSharesOutstanding,us-gaap/2025,20251231,4,shares,NaN,NaN,192246525.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
1264,0000073124-26-000016,WeightedAverageNumberOfDilutedSharesOutstanding,us-gaap/2025,20231231,4,shares,NaN,NaN,207563746.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
1265,0000073124-26-000016,WeightedAverageNumberOfSharesOutstandingBasic,us-gaap/2025,20251231,4,shares,NaN,NaN,191358026.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN
1266,0000073124-26-000016,WeightedAverageNumberOfSharesOutstandingBasic,us-gaap/2025,20231231,4,shares,NaN,NaN,207248094.0,NaN,...,20251231.0,2025.0,FY,20260224,2026-02-24 16:30:00.0,0,1,ntrs-20251231_htm.xml,1,NaN


In [ ]:
import pandas as pd

# 1. Load your original Excel file
# This is your 'Original' (untouched)
df_original = pd.read_excel('NTRS_2026_Master_Data.xlsx')

# 2. Create a "Safety Duplicate" in Jupyter memory
# The .copy() command is critical—it breaks the link to the original
df_cleaning = df_original.copy()

# 3. Immediately save this duplicate as a NEW physical file
# Now you have two files on your Desktop/Folder
df_cleaning.to_excel('NTRS_WORK_IN_PROGRESS.xlsx', index=False)

print("Safety duplicate created! Use 'df_cleaning' for your next tasks.")

Safety duplicate created! Use 'df_cleaning' for your next tasks.


In [ ]:


# 1. Load your original Excel file
try:
    df_original = pd.read_excel('NTRS_2026_Master_Data.xlsx')
    df_cleaning = df_original.copy()

    # 2. Filtering Logic (The 'Elite' Cleaning)
    df_cleaning = df_cleaning[df_cleaning['segments'].isna()] # cite: 5.3
    df_cleaning = df_cleaning[df_cleaning['coreg'].isna()]    # cite: 5.3
    df_cleaning = df_cleaning[df_cleaning['qtrs'].isin([0, 4])] # cite: 5.3

    # 3. Deduplication (Keep latest filed amendment)
    if 'filed' in df_cleaning.columns:
        df_cleaning = df_cleaning.sort_values('filed').drop_duplicates(
            subset=['tag', 'version', 'ddate'], 
            keep='last'
        )

    # 4. Save the Cleaned 'Silver' version to a new file
    output_file = 'NTRS_Cleaned_Financials.xlsx'
    df_cleaning.to_excel(output_file, index=False)

    print("-" * 30)
    print("CLEANING COMPLETE!")
    print(f"New file created: {output_file}")
    print(f"Cleaned rows remaining: {len(df_cleaning)}")
    print(f"Rows removed (noise): {len(df_original) - len(df_cleaning)}")

except Exception as e:
    print(f"An error occurred: {e}. Make sure the Excel file is CLOSED.")

------------------------------
CLEANING COMPLETE!
New file created: NTRS_Cleaned_Financials.xlsx
Cleaned rows remaining: 341
Rows removed (noise): 927


In [ ]:
# Create a Pivot Table: Rows = Accounts, Columns = Dates, Values = Dollars
pivot_df = df_cleaning.pivot_table(
    index=['tag', 'tlabel'], 
    columns='ddate', 
    values='value', 
    aggfunc='first'
).reset_index()

# Save this for your Excel Modeling
pivot_df.to_excel('NTRS_Ready_For_Analysis.xlsx', index=False)
print("Pivot Table Created! Open 'NTRS_Ready_For_Analysis.xlsx' to start the Excel task.")

Pivot Table Created! Open 'NTRS_Ready_For_Analysis.xlsx' to start the Excel task.


In [ ]:
df.head()

NameError: name 'df' is not defined